# Data Exploration

In [ ]:
import pandas as pd
import glob

files = glob.glob("data/raw/*.csv")
print(files)

# Load one file first to see the actual structure
sample = pd.read_csv("data/raw/raw_test_bernie.csv")
print(sample.shape)
print(sample.columns.tolist())
print(sample.head())

In [ ]:
splits = ['train', 'val', 'test']
targets = ['trump', 'biden', 'bernie']

count = 0

for split in splits:
    for target in targets:
        df = pd.read_csv(f"data/raw/raw_{split}_{target}.csv")
        print(f"{split}_{target}: {df.shape[0]} rows")
        print(df['Stance'].value_counts())
        print()

        count += df.shape[0]

print(f"count {count}")

In [ ]:
train_dfs = []
for target in targets:
    df = pd.read_csv(f"data/raw/raw_train_{target}.csv")
    train_dfs.append(df)

train_combined = pd.concat(train_dfs, ignore_index=True)
train_combined.to_csv("data/processed/train_combined.csv", index=False)

In [ ]:
for split in ['train', 'val', 'test']:
    for target in ['trump', 'biden', 'bernie']:
        df = pd.read_csv(f"data/raw/raw_{split}_{target}.csv")
        print(f"{split}_{target}:")
        print(df['Stance'].value_counts(normalize=True))
        print()

In [ ]:
train = pd.read_csv("data/processed/train_combined.csv")
test_dfs = [pd.read_csv(f"data/raw/raw_test_{t}.csv") for t in ['trump','biden','bernie']]
test = pd.concat(test_dfs, ignore_index=True)

overlap = set(train['Tweet']) & set(test['Tweet'])
print(len(overlap))

In [ ]:
from transformers import AutoTokenizer
from src.data import format_prompt

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")

lengths = train.apply(lambda row: len(tokenizer.encode(format_prompt(row['Tweet'], row['Target']))), axis=1)
print(lengths.describe())

# Environment Setup

In [ ]:
# Setup cell - rerun this after any kernel disconnect/reconnect

import os

REPO_DIR = "/content/political-stance-detection"

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    !git clone https://github.com/meghna-adduri/political-stance-detection.git {REPO_DIR}
    %cd {REPO_DIR}

!pip install transformers accelerate bitsandbytes wandb tqdm -q

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
import wandb
import logging

logging.getLogger("bitsandbytes").setLevel(logging.ERROR)

import sys
sys.path.append(REPO_DIR)
from src.data import load_combined, format_prompt
from src.baseline import load_model, get_prediction, parse_label, run_baseline, evaluate_and_log

# Reload model
model, tokenizer = load_model()

# Reload prior results, if they exist
try:
    results = pd.read_csv("data/processed/zeroshot_predictions.csv")
    print(f"Loaded existing results: {len(results)} rows")
except FileNotFoundError:
    print("No existing results file found, run run_baseline() to generate one")

# Confirm GPU is actually attached
!nvidia-smi

In [ ]:
!pwd
!ls

In [ ]:
wandb.login()

In [ ]:
!git config --global user.email "meghna.adduri07@gmail.com"
!git config --global user.name "meghna-adduri"

In [ ]:
!git pull

# Zero-shot Baseline

In [ ]:
from getpass import getpass

results = run_baseline(model, tokenizer)

!git add data/processed/zeroshot_predictions.csv
!git commit -m "Add zero-shot baseline predictions (fixed truncation)"

token = getpass("Enter your GitHub token: ")
!git push https://{token}@github.com/meghna-adduri/political-stance-detection.git main

acc, macro_f1 = evaluate_and_log(results)
print(f"Zero-shot accuracy: {acc:.3f}, macro-F1: {macro_f1:.3f}")

In [ ]:
print(f"Zero-shot accuracy: {acc:.3f}, macro-F1: {macro_f1:.3f}")

In [ ]:
print(results['prediction'].value_counts())

In [ ]:
results[['Tweet', 'Target', 'Stance', 'prediction']].sample(15)

In [ ]:
!git add data/processed/zeroshot_predictions.csv
!git commit -m "Add zero-shot baseline predictions"

# Debugging UNKNOWN predictions

In [ ]:
sample_unknown = results[results['prediction'] == 'UNKNOWN'].sample(15, random_state=42)
sample_favor = results[results['prediction'] == 'FAVOR'].sample(15, random_state=42)
sample_against = results[results['prediction'] == 'AGAINST'].sample(15, random_state=42)

debug_rows = pd.concat([sample_unknown, sample_favor, sample_against])

debug_results = []
for _, row in debug_rows.iterrows():
    raw, pred = get_prediction(model, tokenizer, row['Tweet'], row['Target'])
    debug_results.append({
        'tweet': row['Tweet'][:80],
        'target': row['Target'],
        'original_prediction': row['prediction'],
        'raw_response': raw,
        'response_length_tokens': len(tokenizer.encode(raw))
    })

debug_df = pd.DataFrame(debug_results)

In [ ]:
print(debug_df.groupby('original_prediction')['response_length_tokens'].describe())

In [ ]:
pd.set_option('display.max_colwidth', None)
print(debug_df[['original_prediction', 'raw_response']].to_string())

In [ ]:
print(debug_df['raw_response'].str.strip().str.upper().isin(['FAVOR', 'AGAINST']).mean())

# Useful git commands

In [24]:
from getpass import getpass
token = getpass("Enter your GitHub token: ")
!git push https://{token}@github.com/meghna-adduri/political-stance-detection.git main

Enter your GitHub token: ··········
Everything up-to-date
